# Vision Lab · runner (Kaggle / Colab / local)

One notebook that runs the whole project on a **free GPU**. The code lives in the GitHub repo; this notebook only
clones it, installs the small extras, and calls the scripts.

**Before you run it**

* **Kaggle:** right panel → *Session options* → **Accelerator: GPU T4 x2** (or P100) and **Internet: On**
  (needed to `git clone` and to download the pretrained weights and dataset).
  Use **Save Version → Save & Run All** to train in the background (you can close the tab).
* **Colab:** *Runtime → Change runtime type → T4 GPU*. Checkpoints go to your Google Drive, so a disconnect does not lose the run.
* **Local:** works too (CPU only, so use `--subset 0.02`).

Edit the `REPO_URL` below to point at your GitHub copy of this project.

In [ ]:
REPO_URL = "https://github.com/ali84mlk/vision-lab.git"   # the repo this notebook clones (must be public)
BRANCH = "main"

In [ ]:
import os, sys, subprocess

ON_KAGGLE = os.path.exists("/kaggle/working")
ON_COLAB = "COLAB_GPU" in os.environ or "google.colab" in sys.modules

def sh(*cmd, cwd=None):
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True, cwd=cwd)

# Already inside a copy of the repo (local use)? Then do not clone.
here = os.getcwd()
if os.path.exists(os.path.join(here, "src", "train.py")):
    REPO_DIR = here
elif os.path.exists(os.path.join(os.path.dirname(here), "src", "train.py")):
    REPO_DIR = os.path.dirname(here)
else:
    base = "/kaggle/working" if ON_KAGGLE else ("/content" if ON_COLAB else here)
    REPO_DIR = os.path.join(base, "vision-lab")
    if os.path.exists(REPO_DIR):
        sh("git", "-C", REPO_DIR, "pull", "--ff-only")
    else:
        sh("git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, REPO_DIR)

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print("repo:", REPO_DIR)

In [ ]:
# Colab only: mount Google Drive so checkpoints survive a disconnect (you will be asked to authorise it).
if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

In [ ]:
# torch/torchvision are preinstalled on Kaggle and Colab; this only adds anything missing.
sh(sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt")

import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU: training will be slow. Turn on a GPU accelerator in the session settings.")

## A · Train one model

Loss, optimizer and schedule are the three knobs from the `learn/` notebooks. `--resume` continues from the last
checkpoint if the session was interrupted, so you can re-run this cell safely.

Use `--subset 0.05 --epochs 1` first to check everything works in a minute, then remove `--subset` for the full run.

In [ ]:
from src import train
from src.utils import default_dirs

result = train.main([
    "--dataset", "oxford_pet",
    "--model", "resnet18",
    "--epochs", "10",
    "--optimizer", "adamw",
    "--scheduler", "warmup_cosine",
    "--loss", "label_smoothing",
    "--run-name", "adamw-warmup_cosine",
    "--resume",
])
print("best validation accuracy:", result["best_acc"])

## B · Evaluate it

Overall accuracy, the hardest classes and a confusion matrix (saved next to the checkpoint).

In [ ]:
from src import evaluate
from IPython.display import Image, display

out = default_dirs()["out"]
evaluate.main(["--run-dir", f"{out}/adamw-warmup_cosine"])
display(Image(f"{out}/adamw-warmup_cosine/confusion_matrix.png"))

## C · The learning experiment: compare optimizers and schedules

Same data, same model, same seed; only the optimizer and the LR schedule change. Re-run with `--seeds 0 1 2` to see how
much of a difference is real and how much is noise. Finished runs are skipped if the session restarts.

In [ ]:
from experiments import compare
import pandas as pd
from IPython.display import Image, display

cmp_dir = f"{default_dirs()['out']}/compare"
compare.main([
    "--optimizers", "sgd_momentum", "adamw",
    "--schedulers", "constant", "cosine", "warmup_cosine",
    "--losses", "cross_entropy",
    "--epochs", "5",
    "--out-dir", cmp_dir,
])
display(pd.read_csv(f"{cmp_dir}/summary.csv"))
display(Image(f"{cmp_dir}/curves.png"))

## Notes

* **Kaggle outputs:** everything under `/kaggle/working` is saved with the notebook version (checkpoints, CSVs, PNGs).
  Download `best.pt` from the *Output* tab.
* **Publishing:** on Kaggle use *Share → Public* (the notebook runs from the GitHub repo, so nothing private is in it).
* **Your own data:** put images in `<dir>/<class_name>/*.jpg` and run with `--dataset folder --data-dir <dir>`.